In [299]:
import pandas as pd
import re
import datetime
import ast
import math

## Cleaning Bulan Desember dan November

In [300]:
##Load File Gempa pada bulan November dan Desember tahun 2023 yang masih kotor

des_nov = [
    pd.read_csv('gempa_bumi_desember_dirty.csv'), 
    pd.read_csv('gempa_bumi_desember_dirty.csv')
]

In [301]:
##Untuk menghapus data duplikat yang memiliki string UPDATE dan langsung akan dihapus

for i in range(len(des_nov)):
    des_nov[i] = des_nov[i].rename(columns={'created_at;id_str;full_text;quote_count;reply_count;retweet_count;favorite_count;lang;user_id_str;conversation_id_str;username;tweet_url': 'Unnamed: 0'})
    index = des_nov[i][des_nov[i]['Unnamed: 0'].str.contains('UPDATE')].index
    des_nov[i].drop(index, inplace=True)

des_nov

[                                            Unnamed: 0  \
 0    Sun Dec 31 23:18:11 +0000 2023;174159966158732...   
 1    Sun Dec 31 07:38:09 +0000 2023;174136309376948...   
 2    Sun Dec 31 05:48:08 +0000 2023;174133540656896...   
 4    Sun Dec 31 05:11:40 +0000 2023;174132623155831...   
 5    Sun Dec 31 04:55:37 +0000 2023;174132219405416...   
 ..                                                 ...   
 822  Fri Dec 01 04:22:09 +0000 2023;173044213503454...   
 823  Fri Dec 01 02:35:09 +0000 2023;173041520660885...   
 824  Fri Dec 01 01:46:10 +0000 2023;173040287832879...   
 825  Fri Dec 01 01:11:08 +0000 2023;173039406279104...   
 827  Fri Dec 01 00:36:41 +0000 2023;173038539199444...   
 
                    Unnamed: 1                 Unnamed: 2  \
 0     01-Jan-2024 06:11:23WIB                 Lok:1.89LS   
 1     31-Dec-2023 14:35:34WIB                 Lok:6.84LS   
 2     31-Dec-2023 12:43:38WIB                 Lok:6.74LS   
 4            Kedalaman: 10 km   31 Des 2023 1

In [302]:
clean = []

for i in range(len(des_nov)):
    temp = des_nov[i].drop(['Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10'], axis=1)
    clean.append(temp.dropna())
    
clean

[                                            Unnamed: 0  \
 0    Sun Dec 31 23:18:11 +0000 2023;174159966158732...   
 1    Sun Dec 31 07:38:09 +0000 2023;174136309376948...   
 2    Sun Dec 31 05:48:08 +0000 2023;174133540656896...   
 5    Sun Dec 31 04:55:37 +0000 2023;174132219405416...   
 7    Sun Dec 31 04:55:14 +0000 2023;174132209519445...   
 ..                                                 ...   
 821  Fri Dec 01 04:58:08 +0000 2023;173045119065441...   
 822  Fri Dec 01 04:22:09 +0000 2023;173044213503454...   
 823  Fri Dec 01 02:35:09 +0000 2023;173041520660885...   
 824  Fri Dec 01 01:46:10 +0000 2023;173040287832879...   
 825  Fri Dec 01 01:11:08 +0000 2023;173039406279104...   
 
                    Unnamed: 1    Unnamed: 2  \
 0     01-Jan-2024 06:11:23WIB    Lok:1.89LS   
 1     31-Dec-2023 14:35:34WIB    Lok:6.84LS   
 2     31-Dec-2023 12:43:38WIB    Lok:6.74LS   
 5      31-Des-23 11:52:34 WIB   Lok:8.22 LS   
 7     31-Dec-2023 11:52:33WIB    Lok:8.25LS   
 .

In [303]:
def split_rad(text):
    rad = re.search(r"\((.*?)\)", text)

    if rad is not None:
        temp =  rad.group(1).split()
        return temp[0]
    return "delete"

def split_prov(text):
    prov = re.search(r"\((.*?)\)", text)

    if prov is not None: 
        newText = prov.group(1)
        split = newText.split()
        temp = split[-1].split('-')
        
        return temp[-1]
    return "delete"

def split_mag(text):
    mag = re.search(r"(\d+\.\d+)\s*(Mag(?:nitudo)?)?", text)

    return mag.group(1)

def split_depth(text):   
    depth = re.search(r":\s*(\d+)", str(text))

    return depth.group(1)

def split_lat(text):
    return text.replace("Lok:", "")

def split_lon(text):
    return text.split()[0]

In [305]:
nov_des = pd.DataFrame()

for i in range(len(clean)):
    mag = clean[i][:]["Unnamed: 0"].apply(split_mag)
    depth = clean[i][:]["Unnamed: 4"].apply(split_depth)
    rad = clean[i][:]["Unnamed: 3"].apply(split_rad)
    prov = clean[i][:]["Unnamed: 3"].apply(split_prov)
    lat = clean[i][:]["Unnamed: 2"].apply(split_lat)
    lon = clean[i][:]["Unnamed: 3"].apply(split_lon)
 
    nov_des["mag"] = mag
    nov_des["depth"] = depth
    nov_des["rad"] = rad
    nov_des["prov"] = prov
    nov_des["lat"] = lat
    nov_des["lon"] = lon

nov_des

,mag,depth,rad,prov,lat,lon
0,2.9,10,85,SULTENG,1.89LS,122.53BT
1,4.1,10,2,JABAR,6.84LS,107.93BT
2,2.9,10,27,JABAR,6.74LS,106.60BT
5,5.0,10,90,JABAR,8.22 LS,107.87
7,5.0,10,93,JABAR,8.25LS,107.86BT
...,...,...,...,...,...,...
821,3.2,10,29,ACEHTENGAH,4.54LU,96.59BT
822,2.9,10,107,SULUT,1.78LU,122.85BT
823,4.4,386,187,MALUKUBRTDAYA,6.47LS,127.97BT
824,3.1,24,69,MALUT,1.39LU,126.93BT


## Cleaning Jan-April

In [306]:
def extract_quake_data(text):
    lon = None
    lat = None
    
    magnitude = re.search(r"Mag:(\d+\.\d+)", text).group(1)
    location = re.search(r"\((.+?)\)", text).group(1)
    depth = re.search(r"Kedlmn:(\d+)", text).group(1)
    prov = location.split()[-1].split('-')[-1]
    rad = location.split()[0]
    
    pattern = r"Lok:(\d+\.\d+\w+)\s+(\d+\.\d+)\w+"
    match = re.search(pattern, text)
    
    if match:
        lat = match.group(1)
        
        lon_str = match.group(0)
        lon = lon_str.split(" ")[1]

    quake_data = {
        "mag": magnitude,
        "depth": depth,
        "rad": rad,
        "prov": prov,
        "lat": lat,
        "lon": lon
    }

    return quake_data

In [307]:
month_dirty = [
    pd.read_csv('gempa_bumi_januari.csv'),
    pd.read_csv('gempa_bumi_februari.csv'), 
    pd.read_csv('gempa_bumi_maret.csv'),
    pd.read_csv('gempa_bumi_april.csv')
]

month_clean = [  
]

for i in range(len(month_dirty)):
    cleaned = month_dirty[i][:]["full_text"].apply(extract_quake_data)
    month_clean.append(cleaned)

data_janap = pd.concat(month_clean)
data_janap = pd.DataFrame(data_janap.tolist())
data_janap

,mag,depth,rad,prov,lat,lon
0,3.6,10,74,PAPUABRT,3.92LS,133.71BT
1,2.6,164,3,SUMUT,1.49LU,99.64BT
2,4.5,182,165,MALUKUTENGAH,4.60LS,129.69BT
3,3.2,10,116,JABAR,8.50LS,107.82BT
4,5.0,11,129,MALUT,1.96LU,126.60BT
...,...,...,...,...,...,...
1311,2.9,10,30,SULTENG,2.66LS,121.63BT
1312,3.7,10,137,JATIM,5.80LS,112.60BT
1313,3.0,10,26,SULSEL,3.53LS,120.32BT
1314,3.6,10,115,JATIM,5.96LS,112.49BT


## Gabungkan Dataset

In [328]:
data = pd.concat([data_janap, nov_des])
data = data.dropna()
data

,mag,depth,rad,prov,lat,lon
0,3.6,10,74,PAPUABRT,3.92LS,133.71BT
1,2.6,164,3,SUMUT,1.49LU,99.64BT
2,4.5,182,165,MALUKUTENGAH,4.60LS,129.69BT
3,3.2,10,116,JABAR,8.50LS,107.82BT
4,5.0,11,129,MALUT,1.96LU,126.60BT
...,...,...,...,...,...,...
821,3.2,10,29,ACEHTENGAH,4.54LU,96.59BT
822,2.9,10,107,SULUT,1.78LU,122.85BT
823,4.4,386,187,MALUKUBRTDAYA,6.47LS,127.97BT
824,3.1,24,69,MALUT,1.39LU,126.93BT


## Transformasi Data

In [322]:
def makeLatitude(text):
    lat = float(text[:-2])
    pattern = r"(LS|LU)"
    match = re.search(pattern, text)

    if match:
        if match.group(1) == "LS":
            lat = -1 * float(text[:-2])
        
        return abs(lat) if lat >= 0 else -abs(lat)

def makeLongitude(text):
    lon = float(text[:-2])
    pattern = r"(BT|BB)"
    match = re.search(pattern, text)

    if match:
        if match.group(1) == "BB":
            lon = -1 * float(text[:-2])
        
        return abs(lon) if lon >= 0 else -abs(lon)

In [ ]:
data['lat'] = data[:]['lat'].apply(makeLatitude)
data['lon'] = data[:]['lon'].apply(makeLongitude)

data

In [332]:
data = data.dropna()
data

,mag,depth,rad,prov,lat,lon
0,3.6,10,74,PAPUABRT,-3.92,133.71
1,2.6,164,3,SUMUT,1.49,99.64
2,4.5,182,165,MALUKUTENGAH,-4.60,129.69
3,3.2,10,116,JABAR,-8.50,107.82
4,5.0,11,129,MALUT,1.96,126.60
...,...,...,...,...,...,...
821,3.2,10,29,ACEHTENGAH,4.54,96.59
822,2.9,10,107,SULUT,1.78,122.85
823,4.4,386,187,MALUKUBRTDAYA,-6.47,127.97
824,3.1,24,69,MALUT,1.39,126.93


In [338]:
def encoded_loc(row, encode):
    if row['prov'] in encode:
        return encode[row['prov']]

    return None

unique_locs = data['prov'].unique()
mapLoc = {prov: i + 1 for i, prov in enumerate(unique_locs)}

data['prov_enco'] = data['prov'].map(mapLoc)
data

,mag,depth,rad,prov,lat,lon,prov_enco
0,3.6,10,74,PAPUABRT,-3.92,133.71,1
1,2.6,164,3,SUMUT,1.49,99.64,2
2,4.5,182,165,MALUKUTENGAH,-4.60,129.69,3
3,3.2,10,116,JABAR,-8.50,107.82,4
4,5.0,11,129,MALUT,1.96,126.60,5
...,...,...,...,...,...,...,...
821,3.2,10,29,ACEHTENGAH,4.54,96.59,9
822,2.9,10,107,SULUT,1.78,122.85,15
823,4.4,386,187,MALUKUBRTDAYA,-6.47,127.97,14
824,3.1,24,69,MALUT,1.39,126.93,5


In [340]:
data.to_csv('dataset.csv', index=False)